# All Steps together

## Initialize

In [ ]:
%pip install -U geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.9 MB/s eta 0:00:00


In [ ]:
import ee
import geemap

In [ ]:
ee.Authenticate()
ee.Initialize(project='Project-ID')

In [ ]:
import os
import zipfile
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.ops import unary_union
from tqdm import tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Polyline-Based

In [ ]:
"""
================================================================================
S1S2-Coasts: Coastal Water Boundary Patch Dataset Generator
Stage 1 of 2 — Polyline-Based Coastline Processing (Part-1)
================================================================================

Overview
--------
This module implements the first stage of the S1S2-Coasts patch generation
pipeline, which produces a multi-modal remote sensing dataset of coastal water
boundaries by pairing Sentinel-1 SAR and Sentinel-2 multispectral imagery with
vector-derived binary water boundary masks.

This stage is specifically designed for **polyline input geometries**, such as
coastline shapefiles where the water boundary is already represented as a
LineString (e.g. GSHHG, OpenStreetMap-derived coastlines).

For polygon-based water bodies (e.g. lakes from HydroLakes), refer to Stage 2.

--------------------------------------------------------------------------------
Pipeline Summary
--------------------------------------------------------------------------------
  1. ShapefileProcessor  — Loads the coastline shapefile from a zip archive and
                           stores the LineString geometry for downstream use.

  2. PointGenerator      — Samples N evenly spaced points along the coastline
                           using linear interpolation.

  3. BufferPolygonCreator — Buffers each point into a square patch polygon
                            of configurable size via Google Earth Engine.

  4. WaterBoundaryProcessor — For each patch:
       a. Selects the least-cloudy Sentinel-2 image in the date range.
       b. Finds the temporally closest Sentinel-1 image to the S2 date.
       c. Rasterises the local coastline segment as a binary boundary mask
          (0 = non-water, 255 = water boundary).
       d. Stacks S2 reflectance bands, S1 VV/VH bands, optional spectral
          indices (MNDWI, NDVI, NDPI, WRI, AWI), and the boundary mask
          into a single multi-band GeoTIFF.
       e. Submits a Google Earth Engine batch export task to Google Drive.
       f. Records image IDs and acquisition dates to a metadata CSV.

--------------------------------------------------------------------------------
Input
--------------------------------------------------------------------------------
  - Coastline shapefile (.shp) packaged in a .zip archive
  - Geometry type : LineString (polyline)
  - Example source: GSHHG (Global Self-consistent Hierarchical High-resolution
    Geography), OpenStreetMap coastlines, or equivalent

--------------------------------------------------------------------------------
Output
--------------------------------------------------------------------------------
  - Multi-band GeoTIFF patches exported to Google Drive under:
      Patch_<region_name>/Patch_<region_name>_<i>.tif
  - Metadata CSV saved locally at:
      <output_directory>/<region_name>.csv
    Columns: Patch | S1_image_ID | S2_image_ID | S1_Acquisition_Date |
             S1_Acquisition_Time | S2_Acquisition_Date | S2_Acquisition_Time

--------------------------------------------------------------------------------
Dependencies
--------------------------------------------------------------------------------
  - earthengine-api >= 0.1.350   (Google Earth Engine Python client)
  - geemap >= 0.20.0             (GEE ↔ GeoPandas bridge utilities)
  - geopandas >= 0.12.0
  - shapely >= 2.0.0
  - numpy >= 1.21.0
  - pandas >= 1.3.0
  - tqdm >= 4.62.0

  Install: pip install -r requirements.txt
  GEE auth: earthengine authenticate

--------------------------------------------------------------------------------
Author
--------------------------------------------------------------------------------
  Soheil Zaghian
  Email  : soheil.zaghian.sz@gmail.com
  Stage  : 1 / 2
  Input  : Polyline (coastline)
================================================================================
"""

import os
import zipfile
import geopandas as gpd
import numpy as np
from shapely.ops import unary_union
import pandas as pd
from shapely.geometry import Point, LineString, mapping
from tqdm import tqdm
import logging

# Initialize logging
logging.basicConfig(level=logging.INFO)


class ShapefileProcessor:
    def __init__(self, directory_path, zip_files_directory, output_directory):
        self.directory_path = directory_path
        self.zip_files_directory = zip_files_directory
        self.output_directory = output_directory
        self.coast_lines = []
        self.regions = [f[:-4] for f in os.listdir(zip_files_directory) if f.endswith('.zip')]

    def extract_zip(self, zip_file_path, extract_to_path):
        try:
            os.makedirs(extract_to_path, exist_ok=True)
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(extract_to_path)
            logging.info(f'Contents of {zip_file_path} extracted to {extract_to_path}')
        except Exception as e:
            logging.error(f'Error extracting {zip_file_path}: {e}')

    def load_shapefile(self, file_path):
        try:
            return gpd.read_file(file_path)
        except Exception as e:
            logging.error(f'Error loading shapefile {file_path}: {e}')
            return None

    def process_region(self, region_name):
        region_dir = os.path.join(self.directory_path, region_name)
        print(os.path.exists(region_dir))
        if not os.path.exists(region_dir):
            os.makedirs(region_dir, exist_ok=True)
            zip_file_path = os.path.join(self.zip_files_directory, region_name + '.zip')
            self.extract_zip(zip_file_path, region_dir)
            print('extracted')
        shapefile_path = os.path.join(region_dir, region_name + '.shp')
        shapefile = self.load_shapefile(shapefile_path)
        if shapefile is not None:
            self.coast_lines.append(shapefile)

class PointGenerator:
    @staticmethod
    def generate_points(line, num_points):
        distances = np.linspace(0, line.length, num_points)
        return [line.interpolate(distance) for distance in distances]

class BufferPolygonCreator:
    def __init__(self, patch_size):
        self.patch_size = patch_size

    def create_buffered_polygon(self, feature):
        buffer_distance = self.patch_size / 2
        point = feature.geometry()
        buffer = point.buffer(buffer_distance)
        bounds = buffer.bounds()
        return ee.Feature(bounds).copyProperties(feature)

    def buffer_polygon(self, feature):
        return self.create_buffered_polygon(feature)

class WaterBoundaryProcessor:
    def __init__(self, buffered_polygons, layer, collection_id_sen2, collection_id_sen1, start_date, end_date, output_folder, region_name):
        self.buffered_polygons = buffered_polygons
        self.layer = layer
        self.collection_id_sen2 = collection_id_sen2
        self.collection_id_sen1 = collection_id_sen1
        self.start_date = start_date
        self.end_date = end_date
        self.number_patch = buffered_polygons.size().getInfo()
        self.output_folder = output_folder
        self.region_name = region_name

        # Initialize the variables here to avoid AttributeError if an exception is raised in process_polygon
        self.s1_id = None
        self.s2_id = None
        self.s1_date_only = None
        self.s1_time_only = None
        self.s2_date_only = None
        self.s2_time_only = None
        self.data = []

    @staticmethod
    def create_water_boundary_image(ee_polygon, polygon, layer_gdf):
        polygon = polygon.to_crs(layer_gdf.crs)
        clipped_layer = layer_gdf.clip(polygon.geometry)
        cropped_layer = geemap.gdf_to_ee(clipped_layer)
        zero_image = ee.Image.constant(0)
        painted_image = zero_image.paint(cropped_layer, 255)
        return painted_image.clip(ee_polygon)

    @staticmethod
    def ref_conv(image):
        return image.divide(10000)

    def calculate_date_diff(image, s2_date):
        s1_date = ee.Date(image.get('system:time_start'))
        diff = s1_date.difference(s2_date, 'day').abs()
        return image.set('date_diff', diff)

    @staticmethod
    def calculate_index(image, formula, name):
        index = formula(image).rename(name)
        return image.addBands(index)

    def process_polygon(self, i):
        try:
            feature = ee.Feature(self.buffered_polygons.toList(self.buffered_polygons.size()).get(i))
            ee_polygon = feature.geometry()
            gdf = geemap.ee_to_gdf(ee.FeatureCollection([feature]))
            image = self.create_water_boundary_image(ee_polygon, gdf, self.layer)

            sen2_collection = (
                ee.ImageCollection(self.collection_id_sen2)
                .filterDate(self.start_date, self.end_date)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15))
                .filterBounds(ee_polygon)
                .select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'QA60'])
                # .map(self.ref_conv)
            )

            s2_date = ee.Date(sen2_collection.sort('CLOUDY_PIXEL_PERCENTAGE').first().get('system:time_start'))

            sen2_collection = sen2_collection.map(lambda img: WaterBoundaryProcessor.ref_conv(img))
            indices = [
                (lambda img: img.select('B2').add(img.select('B2').multiply(2.5)).subtract((img.select('B2').add(img.select('B11'))).multiply(1.5)), 'AWI'),
                (lambda img: img.select('B3').add(img.select('B4')).divide(img.select('B8').add(img.select('B11'))), 'WRI'),
                (lambda img: img.select('B8').subtract(img.select('B4')).divide(img.select('B8').add(img.select('B4'))), 'NDVI'),
                (lambda img: img.select('B4').subtract(img.select('B11')).divide(img.select('B4').add(img.select('B11'))), 'NDPI'),
                (lambda img: img.select('B3').subtract(img.select('B11')).divide(img.select('B3').add(img.select('B11'))), 'MNDWI'),
            ]

            for formula, name in indices:
                sen2_collection = sen2_collection.map(lambda img: WaterBoundaryProcessor.calculate_index(img, formula, name))

            image_sen2 = sen2_collection.sort('CLOUDY_PIXEL_PERCENTAGE').first().clip(ee_polygon)
            # print('one')

            # s2_date = ee.Date(image_sen2.get('system:time_start'))

            sen1_collection = (
                ee.ImageCollection(self.collection_id_sen1)
                .filterDate(s2_date.advance(-12, 'month'), s2_date.advance(12, 'month'))
                .filter(ee.Filter.listContains('system:band_names', 'VV'))
                .filter(ee.Filter.listContains('system:band_names', 'VH'))
                .filterBounds(ee_polygon)
            )

            s1_collection_with_diff  = sen1_collection.map(lambda img: WaterBoundaryProcessor.calculate_date_diff(img, s2_date))
            image_sen1 = s1_collection_with_diff.sort('date_diff').first().clip(ee_polygon)

            final_image = image_sen2.addBands(image_sen1).addBands(image.rename('Water_boundary')).toFloat()
            export_params = {
                'image': final_image,
                'description': f'Patch_{self.region_name}_{i}',
                'scale': 10,  # Increase the scale to reduce resolution
                'region': ee_polygon,
                'fileFormat': 'GeoTIFF',
                'folder': f'Patch_{self.region_name}'
            }
            # print('TWO')

            task = ee.batch.Export.image.toDrive(**export_params)
            task.start()

            # Get the acquisition date of the Sentinel-2 image
            s2_date_formatted = s2_date.format().getInfo()
            self.s2_id = image_sen2.get('system:index').getInfo()

            # Get the acquisition date and ID of the closest Sentinel-1 image
            s1_date = ee.Date(image_sen1.get('system:time_start')).format().getInfo()
            self.s1_id = image_sen1.get('system:index').getInfo()

            # Split date and time for Sentinel-1 and Sentinel-2
            self.s1_date_only, self.s1_time_only = s1_date.split('T')
            self.s2_date_only, self.s2_time_only = s2_date_formatted.split('T')

        except Exception as e:
            logging.error(f'Error processing polygon {i}: {e}')

        # return s1_id, s2_id, s1_date_only, s1_time_only, s2_date_only, s2_time_only

    def process_all_polygons(self):
        for i in tqdm(range(self.number_patch), desc="Processing Polygons"):
            self.process_polygon(i)
            self.data.append({
            'Patch': f'Patch_{i+1}',
            # 'Image_ID': closest_s1_id,
            'S1_image_ID': self.s1_id,
            'S2_image_ID': self.s2_id,
            'S1_Acquisition_Date': self.s1_date_only,
            'S1_Acquisition_Time': self.s1_time_only,
            'S2_Acquisition_Date': self.s2_date_only,
            'S2_Acquisition_Time': self.s2_time_only
            })
        pd.DataFrame(self.data).to_csv(f'{self.output_folder}/{self.region_name}.csv', index=False)

def main():
    directory_path = '/content/drive/MyDrive/PhD/WaterBody/'
    zip_files_directory = '/content/drive/MyDrive/WaterBody_ShapeFile_world/'
    output_directory = '/content/drive/MyDrive/Patch-dataset/'
    region_name = 'Madagascar_line'
    shapefile_processor = ShapefileProcessor(directory_path, zip_files_directory, output_directory)

    N_points = 300
    patch_size = 512 * 10
    buffer_creator = BufferPolygonCreator(patch_size)

    # Map = geemap.Map()
    # for region_name in shapefile_processor.regions:

    logging.info(f"Processing region: {region_name}")
    print(f"Processing region: {region_name}")

    shapefile_processor.process_region(region_name)
    coast_line = shapefile_processor.coast_lines[-1]

    line_union = unary_union(coast_line.geometry)
    multipoint = PointGenerator.generate_points(line_union, N_points)
    multipoints = unary_union(multipoint)
    multipoint_gdf = gpd.GeoDataFrame(geometry=[multipoints], crs='4269')
    center_points = multipoint_gdf.explode(index_parts=True).reset_index(drop=True)
    center_points_ee = geemap.gdf_to_ee(center_points)
    buffered_polygons = center_points_ee.map(buffer_creator.buffer_polygon)

    # print(buffered_polygons.size().getInfo())
    layer = coast_line
    collection_id_sen2 = "COPERNICUS/S2_SR_HARMONIZED"
    collection_id_sen1 = "COPERNICUS/S1_GRD"
    start_date = '2021-01-01'
    end_date = '2024-12-30'

    processor = WaterBoundaryProcessor(buffered_polygons, layer, collection_id_sen2, collection_id_sen1, start_date, end_date, output_directory, region_name)
    processor.process_all_polygons()

if __name__ == "__main__":
    main()

In [ ]:
"""
For a single region example

"""

# directory_path = '/content/drive/MyDrive/WaterBody/'
# zip_files_directory = '/content/drive/MyDrive/WaterBody_ShapeFile_world/OSM'
# output_directory = '/content/drive/MyDrive/Patch-dataset/'
# region_name = 'Madagascar_line'
# shapefile_processor = ShapefileProcessor(directory_path, zip_files_directory, output_directory)

# N_points = 300
# patch_size = 700 * 10
# buffer_creator = BufferPolygonCreator(patch_size)

# # Map = geemap.Map()
# # for region_name in shapefile_processor.regions:

# logging.info(f"Processing region: {region_name}")
# print(f"Processing region: {region_name}")
# #
# shapefile_processor.process_region(region_name)
# coast_line = shapefile_processor.coast_lines[-1]

# line_union = unary_union(coast_line.geometry)
# multipoint = PointGenerator.generate_points(line_union, N_points)
# multipoints = unary_union(multipoint)
# multipoint_gdf = gpd.GeoDataFrame(geometry=[multipoints], crs='4269')
# center_points = multipoint_gdf.explode(index_parts=True).reset_index(drop=True)
# center_points_ee = geemap.gdf_to_ee(center_points)
# buffered_polygons = center_points_ee.map(buffer_creator.buffer_polygon)



## polygon-Based

In [ ]:
"""
================================================================================
S1S2-Coasts: Lake Water Boundary Patch Dataset Generator
Stage 2 of 2 — Polygon-Based Lake Boundary Processing
================================================================================

Overview
--------
This module implements the second stage of the S1S2-Coasts patch generation
pipeline, targeting **polygon input geometries** that represent enclosed water
body boundaries, such as lake shorelines from the HydroLakes dataset.

Unlike Stage 1 (coastlines as LineStrings), lake boundaries are stored as
closed Polygons. This stage handles that distinction by extracting the exterior
ring of each polygon as a LineString for point sampling, while preserving the
original polygon as the water boundary label layer for rasterisation.

For polyline-based coastlines (e.g. GSHHG, OpenStreetMap), refer to Stage 1.

--------------------------------------------------------------------------------
Pipeline Summary
--------------------------------------------------------------------------------
  1. ShapefileProcessor  — Loads the lake boundary shapefile from a zip archive.
                           Detects the Polygon geometry and extracts exterior
                           rings as LineStrings for point sampling, while keeping
                           the original Polygon layer for boundary labelling.

  2. PointGenerator      — Samples N evenly spaced points along the extracted
                           exterior LineString using linear interpolation.

  3. BufferPolygonCreator — Buffers each point into a square patch polygon
                            of configurable size via Google Earth Engine.

  4. WaterBoundaryProcessor — For each patch:
       a. Selects the least-cloudy Sentinel-2 image in the date range.
       b. Finds the temporally closest Sentinel-1 image to the S2 date,
          using a wider backward search window (up to 24 months) to account
          for SAR coverage gaps over inland water bodies.
       c. Clips the original lake Polygon to the patch extent and rasterises
          it as a binary boundary mask (0 = non-water, 255 = water boundary).
       d. Stacks S2 reflectance bands, S1 VV/VH bands, and the boundary mask
          into a single multi-band GeoTIFF.
       e. Submits a Google Earth Engine batch export task to Google Drive.
       f. Records image IDs and acquisition dates to a metadata CSV.

--------------------------------------------------------------------------------
Input
--------------------------------------------------------------------------------
  - Lake boundary shapefile (.shp) packaged in a .zip archive
  - Geometry type : Polygon (closed lake boundary)
  - Example source: HydroLakes v1.0 (Messager et al., 2016)
                    https://www.hydrosheds.org/products/hydrolakes
  - Polygon exterior ring is extracted internally — no pre-processing required

--------------------------------------------------------------------------------
Output
--------------------------------------------------------------------------------
  - Multi-band GeoTIFF patches exported to Google Drive under:
      Patch_<region_name>/Patch_<region_name>_<i>.tif
  - Metadata CSV saved locally at:
      <output_directory>/<region_name>.csv
    Columns: Patch | S1_image_ID | S2_image_ID | S1_Acquisition_Date |
             S1_Acquisition_Time | S2_Acquisition_Date | S2_Acquisition_Time

--------------------------------------------------------------------------------
Key Differences from Stage 1 (Polyline / Coastline)
--------------------------------------------------------------------------------
  Feature                  Stage 1 - Part 1 (Polyline)  Stage 1 - Part 2 (Polygon)
  ─────────────────────────────────────────────────────────────────────
  Input geometry           LineString                   Polygon
  Geometry conversion      None                         Exterior ring → LineString
  Label layer              Original linestring          Original polygon
  S1 backward search       12 months                    24 months
  Cloud threshold          15 %                         20 %
  Example region           Madagascar coastline         Lake Huron
  Example data source      GSHHG / OSM coastlines       HydroLakes v1.0

--------------------------------------------------------------------------------
Dependencies
--------------------------------------------------------------------------------
  - earthengine-api >= 0.1.350   (Google Earth Engine Python client)
  - geemap >= 0.20.0             (GEE ↔ GeoPandas bridge utilities)
  - geopandas >= 0.12.0
  - shapely >= 2.0.0
  - numpy >= 1.21.0
  - pandas >= 1.3.0
  - tqdm >= 4.62.0

  Install: pip install -r requirements.txt
  GEE auth: earthengine authenticate

--------------------------------------------------------------------------------
Author
--------------------------------------------------------------------------------
  Soheil Zaghian
  Email  : soheil.zaghian.sz@gmail.com
  Stage  : 1 / 2
  Input  : Polygon (lake boundary)
================================================================================
"""


import os
import zipfile
import geopandas as gpd
import numpy as np
from shapely.ops import unary_union
import pandas as pd
from shapely.geometry import Point, LineString, mapping
from tqdm import tqdm
import logging

# Initialize logging
logging.basicConfig(level=logging.INFO)


class ShapefileProcessor:
    def __init__(self, directory_path, zip_files_directory, output_directory):
        self.directory_path = directory_path
        self.zip_files_directory = zip_files_directory
        self.output_directory = output_directory
        self.coast_lines = []
        self.regions = [f[:-4] for f in os.listdir(zip_files_directory) if f.endswith('.zip')]

    def extract_zip(self, zip_file_path, extract_to_path):
        try:
            os.makedirs(extract_to_path, exist_ok=True)
            with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                zip_ref.extractall(extract_to_path)
            logging.info(f'Contents of {zip_file_path} extracted to {extract_to_path}')
        except Exception as e:
            logging.error(f'Error extracting {zip_file_path}: {e}')

    def load_shapefile(self, file_path):
        try:
            return gpd.read_file(file_path)
        except Exception as e:
            logging.error(f'Error loading shapefile {file_path}: {e}')
            return None

    def process_region(self, region_name):
        region_dir = os.path.join(self.directory_path, region_name)
        print(os.path.exists(region_dir))
        if not os.path.exists(region_dir):
            os.makedirs(region_dir, exist_ok=True)
            zip_file_path = os.path.join(self.zip_files_directory, region_name + '.zip')
            self.extract_zip(zip_file_path, region_dir)
            print('extracted')
        shapefile_path = os.path.join(region_dir, region_name + '.shp')
        shapefile = self.load_shapefile(shapefile_path)
        if shapefile is not None:
            self.coast_lines.append(shapefile)

class PointGenerator:
    @staticmethod
    def generate_points(line, num_points):
        distances = np.linspace(0, line.length, num_points)
        return [line.interpolate(distance) for distance in distances]

class BufferPolygonCreator:
    def __init__(self, patch_size):
        self.patch_size = patch_size

    def create_buffered_polygon(self, feature):
        buffer_distance = self.patch_size / 2
        point = feature.geometry()
        buffer = point.buffer(buffer_distance)
        bounds = buffer.bounds()
        return ee.Feature(bounds).copyProperties(feature)

    def buffer_polygon(self, feature):
        return self.create_buffered_polygon(feature)

class WaterBoundaryProcessor:
    def __init__(self, buffered_polygons, layer, collection_id_sen2, collection_id_sen1, start_date, end_date, output_folder, region_name):
        self.buffered_polygons = buffered_polygons
        self.layer = layer
        self.collection_id_sen2 = collection_id_sen2
        self.collection_id_sen1 = collection_id_sen1
        self.start_date = start_date
        self.end_date = end_date
        self.number_patch = buffered_polygons.size().getInfo()
        self.output_folder = output_folder
        self.region_name = region_name

        # Initialize the variables here to avoid AttributeError if an exception is raised in process_polygon
        self.s1_id = None
        self.s2_id = None
        self.s1_date_only = None
        self.s1_time_only = None
        self.s2_date_only = None
        self.s2_time_only = None
        self.data = []

    @staticmethod
    def create_water_boundary_image(ee_polygon, polygon, layer_gdf):
        polygon = polygon.to_crs(layer_gdf.crs)
        clipped_layer = layer_gdf.clip(polygon.geometry)
        cropped_layer = geemap.gdf_to_ee(clipped_layer)
        zero_image = ee.Image.constant(0)
        painted_image = zero_image.paint(cropped_layer, 255)
        return painted_image.clip(ee_polygon)

    @staticmethod
    def ref_conv(image):
        return image.divide(10000)

    def calculate_date_diff(image, s2_date):
        s1_date = ee.Date(image.get('system:time_start'))
        diff = s1_date.difference(s2_date, 'day').abs()
        return image.set('date_diff', diff)

    @staticmethod
    def calculate_index(image, formula, name):
        index = formula(image).rename(name)
        return image.addBands(index)

    def process_polygon(self, i):
        try:
            feature = ee.Feature(buffered_polygons.toList(buffered_polygons.size()).get(i))
            ee_polygon = feature.geometry()
            gdf = geemap.ee_to_gdf(ee.FeatureCollection([feature]))
            image = WaterBoundaryProcessor.create_water_boundary_image(ee_polygon, gdf, layer)

            sen2_collection = (
                ee.ImageCollection(collection_id_sen2)
                .filterDate(start_date, end_date)
                .filterBounds(ee_polygon)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
                .select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'QA60'])
                # .map(self.ref_conv)
            )

            s2_date = ee.Date(sen2_collection.sort('CLOUDY_PIXEL_PERCENTAGE').first().get('system:time_start'))

            # sen2_collection = sen2_collection.map(lambda img: WaterBoundaryProcessor.ref_conv(img))
            # indices = [
            #     (lambda img: img.select('B2').add(img.select('B2').multiply(2.5)).subtract((img.select('B2').add(img.select('B11'))).multiply(1.5)), 'AWI'),
            #     (lambda img: img.select('B3').add(img.select('B4')).divide(img.select('B8').add(img.select('B11'))), 'WRI'),
            #     (lambda img: img.select('B8').subtract(img.select('B4')).divide(img.select('B8').add(img.select('B4'))), 'NDVI'),
            #     (lambda img: img.select('B4').subtract(img.select('B11')).divide(img.select('B4').add(img.select('B11'))), 'NDPI'),
            #     (lambda img: img.select('B3').subtract(img.select('B11')).divide(img.select('B3').add(img.select('B11'))), 'MNDWI'),
            # ]

            # for formula, name in indices:
            #     sen2_collection = sen2_collection.map(lambda img: WaterBoundaryProcessor.calculate_index(img, formula, name))

            image_sen2 = sen2_collection.sort('CLOUDY_PIXEL_PERCENTAGE').first().clip(ee_polygon)
            # # print('one')
            # image_sen2


            sen1_collection = (
              ee.ImageCollection(collection_id_sen1)
              .filterDate(s2_date.advance(-24, 'month'), s2_date.advance(12, 'month'))
              .filter(ee.Filter.listContains('system:band_names', 'VV'))
              .filter(ee.Filter.listContains('system:band_names', 'VH'))
              .filterBounds(ee_polygon)
              .select(['VV', 'VH'])

            )

            s1_collection_with_diff  = sen1_collection.map(lambda img: WaterBoundaryProcessor.calculate_date_diff(img, s2_date))
            image_sen1 = s1_collection_with_diff.sort('date_diff').first().clip(ee_polygon)

            final_image = image_sen2.addBands(image_sen1).addBands(image.rename('Water_boundary')).toFloat()
            export_params = {
                'image': final_image,
                'description': f'Patch_{self.region_name}_{i}',
                'scale': 10,  # Increase the scale to reduce resolution
                'region': ee_polygon,
                'fileFormat': 'GeoTIFF',
                'folder': f'Patch_{self.region_name}'
            }
            # print('TWO')

            task = ee.batch.Export.image.toDrive(**export_params)
            task.start()

            # Get the acquisition date of the Sentinel-2 image
            s2_date_formatted = s2_date.format().getInfo()
            self.s2_id = image_sen2.get('system:index').getInfo()

            # Get the acquisition date and ID of the closest Sentinel-1 image
            s1_date = ee.Date(image_sen1.get('system:time_start')).format().getInfo()
            self.s1_id = image_sen1.get('system:index').getInfo()

            # Split date and time for Sentinel-1 and Sentinel-2
            self.s1_date_only, self.s1_time_only = s1_date.split('T')
            self.s2_date_only, self.s2_time_only = s2_date_formatted.split('T')

        except Exception as e:
            logging.error(f'Error processing polygon {i}: {e}')

        # return s1_id, s2_id, s1_date_only, s1_time_only, s2_date_only, s2_time_only

    def process_all_polygons(self):
        for i in tqdm(range(self.number_patch), desc="Processing Polygons"):
            self.process_polygon(i)
            self.data.append({
            'Patch': f'Patch_{i+1}',
            # 'Image_ID': closest_s1_id,
            'S1_image_ID': self.s1_id,
            'S2_image_ID': self.s2_id,
            'S1_Acquisition_Date': self.s1_date_only,
            'S1_Acquisition_Time': self.s1_time_only,
            'S2_Acquisition_Date': self.s2_date_only,
            'S2_Acquisition_Time': self.s2_time_only
            })
        pd.DataFrame(self.data).to_csv(f'{self.output_folder}/{self.region_name}.csv', index=False)
def main():
      directory_path = '/content/drive/MyDrive/PhD/WaterBody/'
      zip_files_directory = '/content/drive/MyDrive/PhD/WaterBody/HydroLakes/'
      output_directory = '/content/drive/MyDrive/Patch-dataset/'
      region_name = 'Huron'
      shapefile_processor = ShapefileProcessor(directory_path, zip_files_directory, output_directory)

      N_points = 200
      patch_size = 512 * 10
      buffer_creator = BufferPolygonCreator(patch_size)


      logging.info(f"Processing region: {region_name}")
      print(f"Processing region: {region_name}")

      shapefile_processor.process_region(region_name)
      coast_line = shapefile_processor.coast_lines[-1]

      coast_poly = coast_line.copy()


      #
      polygon = coast_line.geometry[0]
      polyline_coords = list(polygon.exterior.coords[:-1])  # Exclude the last point to avoid closing the line
      polyline = LineString(polyline_coords)

      coast_line['geometry']= polyline

      #
      line_union = unary_union(coast_line.geometry)
      multipoint = PointGenerator.generate_points(line_union, N_points)
      multipoints = unary_union(multipoint)
      multipoint_gdf = gpd.GeoDataFrame(geometry=[multipoints], crs='4269')
      center_points = multipoint_gdf.explode(index_parts=True).reset_index(drop=True)
      center_points_ee = geemap.gdf_to_ee(center_points)
      buffered_polygons = center_points_ee.map(buffer_creator.buffer_polygon)
      print(buffered_polygons.size().getInfo())
      layer = coast_poly
      collection_id_sen2 = "COPERNICUS/S2_SR_HARMONIZED"
      collection_id_sen1 = "COPERNICUS/S1_GRD"
      start_date = '2019-01-01'
      end_date = '2023-12-30'

      # processor = WaterBoundaryProcessor(buffered_polygons, layer, collection_id_sen2, collection_id_sen1, start_date, end_date, output_directory, region_name)
      # processor.process_all_polygons()

if __name__ == "__main__":
    main()

Processing region: Huron
True
200


In [ ]:

"""
For a single region example

"""


# directory_path = '/content/drive/MyDrive/PhD/WaterBody/'
# zip_files_directory = '/content/drive/MyDrive/PhD/WaterBody/HydroLakes/'
# output_directory = '/content/drive/MyDrive/Patch-dataset/'
# region_name = 'Great Slave'
# shapefile_processor = ShapefileProcessor(directory_path, zip_files_directory, output_directory)

# N_points = 200
# patch_size = 700 * 10
# buffer_creator = BufferPolygonCreator(patch_size)


# logging.info(f"Processing region: {region_name}")
# print(f"Processing region: {region_name}")

# shapefile_processor.process_region(region_name)
# coast_line = shapefile_processor.coast_lines[-1]

# coast_poly = coast_line.copy()


# #
# polygon = coast_line.geometry[0]
# polyline_coords = list(polygon.exterior.coords[:-1])  # Exclude the last point to avoid closing the line
# polyline = LineString(polyline_coords)

# coast_line['geometry']= polyline

# #
# line_union = unary_union(coast_line.geometry)
# multipoint = PointGenerator.generate_points(line_union, N_points)
# multipoints = unary_union(multipoint)
# multipoint_gdf = gpd.GeoDataFrame(geometry=[multipoints], crs='4269')
# center_points = multipoint_gdf.explode(index_parts=True).reset_index(drop=True)
# center_points_ee = geemap.gdf_to_ee(center_points)
# buffered_polygons = center_points_ee.map(buffer_creator.buffer_polygon)
# print(buffered_polygons.size().getInfo())
# layer = coast_poly
# collection_id_sen2 = "COPERNICUS/S2_SR_HARMONIZED"
# collection_id_sen1 = "COPERNICUS/S1_GRD"
# start_date = '2019-01-01'
# end_date = '2023-12-30'


# Map = geemap.Map()
# Map.addLayer(buffered_polygons)
# Map.centerObject(buffered_polygons, zoom=7)
# Map
# processor = WaterBoundaryProcessor(buffered_polygons, layer, collection_id_sen2, collection_id_sen1, start_date, end_date, output_directory, region_name)
# processor.process_all_polygons()